# Cómputo Vectorizado en Física: De los Bucles a NumPy

*Métodos Computacionales Modernos para la Física — Módulo II: Fundamentos de Alto Desempeño*

La Sesión 6 del curso ya diseccionó **por qué** NumPy es rápido (memoria contigua,
`strides`, ufuncs, broadcasting) usando como caso de estudio el cuello de botella
real de `RelicSolver`. Este cuaderno complementa esa sesión con un recorrido más
amplio, ejecutado y medido en vivo con `%timeit`, sobre **tareas numéricas que
aparecen constantemente en física computacional**: evaluar una función sobre una
rejilla, sumar sobre muchos estados, calcular interacciones por pares entre $N$
partículas, y — el punto que casi siempre se entiende mal — qué gana realmente
`numpy.vectorize` y qué no.

La regla del curso se mantiene: no se teoriza sobre el desempeño, se mide.

In [1]:
import sys, math, timeit
import numpy as np
import scipy
from scipy import special

print(f"Python  {sys.version.split()[0]}")
print(f"NumPy   {np.__version__}")
print(f"SciPy   {scipy.__version__}")

# Aquí acumulamos los tiempos promedio (en segundos) de cada experimento,
# para armar una tabla comparativa al final del cuaderno.
resultados = {}

Python  3.12.12
NumPy   2.4.2
SciPy   1.17.1


## 1. La herramienta: `%timeit`

`%timeit` ejecuta una expresión muchas veces (auto-ajustando cuántas para que
la medición dome unas décimas de segundo), repite ese bloque varias veces, y
reporta el mejor promedio — evitando que un solo evento aleatorio del sistema
operativo contamine la medición. Con la bandera `-o` el resultado se puede
capturar en una variable de Python (`t.average` en segundos), lo cual usaremos
para construir la tabla de la Sección 6 sin copiar números a mano.

Ejemplo mínimo: elevar al cuadrado un millón de números, con una lista de
Python por comprensión contra un arreglo de NumPy.

In [2]:
N = 1_000_000
x_list = list(range(N))
x_arr = np.arange(N)

In [3]:
t_loop = %timeit -o [xi**2 for xi in x_list]

24.3 ms ± 888 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [4]:
t_vec = %timeit -o x_arr**2

253 μs ± 16.8 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [5]:
resultados['Cuadrado de 1e6 elementos — lista Python'] = t_loop.average
resultados['Cuadrado de 1e6 elementos — NumPy'] = t_vec.average
print(f"Ganancia: {t_loop.average / t_vec.average:,.0f}x")

Ganancia: 96x


## 2. Tarea física 1 — Evaluar una función sobre una rejilla: distribución de Maxwell–Boltzmann

Una tarea que aparece en casi cualquier curso de física estadística o cinética
de gases: evaluar la distribución de rapideces de Maxwell–Boltzmann

$$f(v) = 4\pi\left(\frac{m}{2\pi k_BT}\right)^{3/2} v^2\, e^{-mv^2/2k_BT}$$

sobre una rejilla fina de velocidades (para graficarla, integrarla, o
muestrear de ella). Usamos unidades reducidas $m = k_BT = 1$; el resultado
generaliza por un simple reescalamiento.

In [6]:
def f_mb_escalar(v, m=1.0, kT=1.0):
    return 4*math.pi*(m/(2*math.pi*kT))**1.5 * v**2 * math.exp(-m*v**2/(2*kT))

def f_mb_bucle(v_array):
    return [f_mb_escalar(v) for v in v_array]

def f_mb_vectorizada(v, m=1.0, kT=1.0):
    return 4*np.pi*(m/(2*np.pi*kT))**1.5 * v**2 * np.exp(-m*v**2/(2*kT))

v_grid = np.linspace(0.0, 10.0, 300_000)
v_grid_lista = v_grid.tolist()  # bucle sobre floats de Python, no sobre el arreglo

In [7]:
# Validación física antes de medir nada: la distribución debe normalizar a 1
# (truncada en v=10, muy por encima de donde f(v) ya es despreciable).
norma = np.trapezoid(f_mb_vectorizada(v_grid), v_grid)
assert np.isclose(norma, 1.0, rtol=1e-3), norma

# Consistencia bucle vs. vectorizado
assert np.allclose(f_mb_bucle(v_grid_lista), f_mb_vectorizada(v_grid))
print(f"Normalización numérica: {norma:.6f} (esperado 1.0)")

Normalización numérica: 1.000000 (esperado 1.0)


In [8]:
t_mb_loop = %timeit -o f_mb_bucle(v_grid_lista)

47.3 ms ± 408 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [9]:
t_mb_vec = %timeit -o f_mb_vectorizada(v_grid)

1.16 ms ± 2.57 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [10]:
resultados['Maxwell-Boltzmann sobre 3e5 puntos — bucle'] = t_mb_loop.average
resultados['Maxwell-Boltzmann sobre 3e5 puntos — NumPy'] = t_mb_vec.average
print(f"Ganancia: {t_mb_loop.average / t_mb_vec.average:,.0f}x")

Ganancia: 41x


## 3. Tarea física 2 — Suma sobre muchos estados: función de partición canónica

Otra tarea recurrente: sumar sobre un número grande de microestados o niveles
de energía para construir la función de partición canónica,

$$Z(\beta) = \sum_i e^{-\beta E_i}.$$

Aquí la "rejilla" no es espacial sino un espectro de energías — el mismo patrón
algebraico (aplicar `exp` a cada elemento y sumar) domina también el cálculo de
$\langle E\rangle = -\partial_\beta \ln Z$ y de $\sigma_E^2$ que ya aparecieron
en los exámenes de Física Estadística.

In [11]:
E_niveles = np.linspace(0.0, 50.0, 300_000)
E_niveles_lista = E_niveles.tolist()
beta = 1.0

def Z_bucle(E, beta):
    total = 0.0
    for e in E:
        total += math.exp(-beta*e)
    return total

def Z_vectorizada(E, beta):
    return np.sum(np.exp(-beta*E))

assert np.isclose(Z_bucle(E_niveles_lista, beta), Z_vectorizada(E_niveles, beta))

In [12]:
t_Z_loop = %timeit -o Z_bucle(E_niveles_lista, beta)

7.16 ms ± 31.7 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [13]:
t_Z_vec = %timeit -o Z_vectorizada(E_niveles, beta)

885 μs ± 34.7 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [14]:
resultados['Función de partición, 3e5 niveles — bucle'] = t_Z_loop.average
resultados['Función de partición, 3e5 niveles — NumPy'] = t_Z_vec.average
print(f"Ganancia: {t_Z_loop.average / t_Z_vec.average:,.0f}x")

Ganancia: 8x


## 4. Tarea física 3 — Interacción por pares en un sistema de $N$ partículas

El ejemplo más clásico de todos: energía potencial gravitacional (o
Coulombiana, la forma algebraica es la misma) de un sistema de $N$ partículas,

$$U = -G\sum_{i<j} \frac{m_i m_j}{r_{ij}}, \qquad r_{ij} = |\vec r_i - \vec r_j|.$$

La versión ingenua tiene dos bucles anidados — $O(N^2)$ operaciones — **y
además indexa un arreglo de NumPy elemento por elemento dentro del bucle**, lo
cual es más lento todavía que indexar una lista de Python: cada `pos[i, 0]`
empaqueta un `float` de NumPy en un objeto de Python nuevo. La versión
vectorizada reemplaza los dos bucles por *broadcasting*: `pos[:, None, :] -
pos[None, :, :]` construye de una vez el arreglo $(N,N,3)$ de todas las
diferencias por pares.

Esa construcción tiene un costo que no hay que esconder: el arreglo de
distancias es $O(N^2)$ en memoria, no $O(N)$. Para $N$ de miles en adelante,
esta estrategia deja de caber en memoria antes de que deje de ser la más
rápida — el mismo compromiso *memory-bound* vs. *compute-bound* de la Sesión
5, aquí aplicado al espacio en vez de al tiempo.

In [15]:
rng = np.random.default_rng(42)
N = 400
pos = rng.uniform(-1.0, 1.0, size=(N, 3))
masas = rng.uniform(0.5, 2.0, size=N)
G = 1.0

def potencial_bucle(pos, masas, G):
    N = len(masas)
    U = 0.0
    for i in range(N):
        for j in range(i+1, N):
            dx = pos[i, 0] - pos[j, 0]
            dy = pos[i, 1] - pos[j, 1]
            dz = pos[i, 2] - pos[j, 2]
            r = math.sqrt(dx*dx + dy*dy + dz*dz)
            U -= G*masas[i]*masas[j]/r
    return U

def potencial_vectorizado(pos, masas, G):
    diff = pos[:, None, :] - pos[None, :, :]          # (N, N, 3)
    r = np.sqrt(np.sum(diff**2, axis=-1))              # (N, N)
    np.fill_diagonal(r, np.inf)                        # evita 1/0 en i == j
    U_pares = -G * np.outer(masas, masas) / r
    return 0.5 * np.sum(U_pares)                        # cada par se cuenta dos veces

assert np.isclose(potencial_bucle(pos, masas, G), potencial_vectorizado(pos, masas, G))

In [16]:
# La versión con bucles es lenta: pocas repeticiones para no esperar de más.
t_pot_loop = %timeit -o -n 1 -r 5 potencial_bucle(pos, masas, G)

51.4 ms ± 364 μs per loop (mean ± std. dev. of 5 runs, 1 loop each)


In [17]:
t_pot_vec = %timeit -o potencial_vectorizado(pos, masas, G)

2.78 ms ± 12.1 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [18]:
resultados[f'Potencial por pares, N={N} — bucle'] = t_pot_loop.average
resultados[f'Potencial por pares, N={N} — NumPy (broadcasting)'] = t_pot_vec.average
print(f"Ganancia: {t_pot_loop.average / t_pot_vec.average:,.0f}x")

Ganancia: 18x


## 5. `numpy.vectorize`: conveniencia, no verdadera vectorización

Esta es la trampa conceptual más común del tema. `numpy.vectorize` existe para
aplicar sobre un arreglo una función que solo sabe operar sobre escalares — por
ejemplo, `math.erf`, la función error, que aparece en física cada vez que se
integra una gaussiana (la fracción acumulada de partículas con rapidez menor a
$v$ en una distribución de Maxwell–Boltzmann, entre muchos otros casos).
`math.erf` no acepta un arreglo de NumPy directamente.

La tentación es pensar que envolverla con `np.vectorize` la vuelve tan rápida
como una ufunc real. **No es así**: por dentro, `np.vectorize` sigue siendo un
bucle de Python, uno por elemento — la propia documentación de NumPy lo dice
explícitamente: *"The vectorize function is provided primarily for
convenience, not for performance. The implementation is essentially a for
loop."* Lo comparamos con tres estrategias:

1. un bucle explícito de Python llamando a `math.erf`,
2. `math.erf` envuelta con `np.vectorize`,
3. `scipy.special.erf`, que sí es una ufunc genuina, compilada en C.

In [19]:
x_grid = np.linspace(0.01, 5.0, 200_000)
x_grid_lista = x_grid.tolist()

def erf_bucle(x_array):
    return [math.erf(x) for x in x_array]

erf_vectorize = np.vectorize(math.erf)

assert np.allclose(erf_bucle(x_grid_lista), erf_vectorize(x_grid))
assert np.allclose(erf_bucle(x_grid_lista), special.erf(x_grid))

In [20]:
t_erf_loop = %timeit -o erf_bucle(x_grid_lista)

5.32 ms ± 222 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [21]:
t_erf_vectorize = %timeit -o erf_vectorize(x_grid)

13.5 ms ± 23.8 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [22]:
t_erf_scipy = %timeit -o special.erf(x_grid)

1.37 ms ± 2.72 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [23]:
resultados['erf sobre 2e5 puntos — bucle Python'] = t_erf_loop.average
resultados['erf sobre 2e5 puntos — np.vectorize'] = t_erf_vectorize.average
resultados['erf sobre 2e5 puntos — scipy.special.erf (ufunc real)'] = t_erf_scipy.average

print(f"np.vectorize vs. bucle explícito : {t_erf_loop.average / t_erf_vectorize.average:5.2f}x")
print(f"scipy.special.erf vs. bucle       : {t_erf_loop.average / t_erf_scipy.average:8.0f}x")
print(f"scipy.special.erf vs. np.vectorize: {t_erf_vectorize.average / t_erf_scipy.average:8.0f}x")

np.vectorize vs. bucle explícito :  0.40x
scipy.special.erf vs. bucle       :        4x
scipy.special.erf vs. np.vectorize:       10x


Los números, medidos arriba y no supuestos, son incluso más contundentes de
lo que promete la documentación: en esta corrida, `np.vectorize` no solo no
aceleró nada frente al bucle explícito — fue **más lento** (el bucle de Python
sigue ahí, y encima carga el overhead de la maquinaria de broadcasting de
`np.vectorize` alrededor de cada llamada escalar). La ufunc real de SciPy, en
cambio, sí fue varias veces más rápida tanto que el bucle como que
`np.vectorize` — la diferencia entre "quitar el bucle de la vista" y "quitar
el bucle de verdad". La lección para el resto del curso: si existe una ufunc
real para la operación que se necesita (NumPy, SciPy, o una escrita a mano con
Numba en la Sesión 7), úsala; `np.vectorize` sirve para prototipar rápido o
para aplicar broadcasting a una función que de plano no tiene una versión
vectorizada disponible — nunca como estrategia de desempeño, y en este
experimento concreto, ni siquiera como mejora incidental.

## 6. Resumen de lo medido

Todo lo anterior se ejecutó de verdad, en esta corrida, en este entorno —
nada de la tabla siguiente está escrito a mano.

In [24]:
ancho = max(len(k) for k in resultados)
print(f"{'Experimento':<{ancho}}  {'t promedio':>14}")
print('-' * (ancho + 16))
for nombre, t in resultados.items():
    if t < 1e-3:
        texto = f"{t*1e6:9.1f} µs"
    elif t < 1.0:
        texto = f"{t*1e3:9.2f} ms"
    else:
        texto = f"{t:9.3f} s "
    print(f"{nombre:<{ancho}}  {texto:>14}")

Experimento                                                t promedio
---------------------------------------------------------------------
Cuadrado de 1e6 elementos — lista Python                     24.29 ms
Cuadrado de 1e6 elementos — NumPy                            253.1 µs
Maxwell-Boltzmann sobre 3e5 puntos — bucle                   47.28 ms
Maxwell-Boltzmann sobre 3e5 puntos — NumPy                    1.16 ms
Función de partición, 3e5 niveles — bucle                     7.16 ms
Función de partición, 3e5 niveles — NumPy                    885.3 µs
Potencial por pares, N=400 — bucle                           51.40 ms
Potencial por pares, N=400 — NumPy (broadcasting)             2.78 ms
erf sobre 2e5 puntos — bucle Python                           5.32 ms
erf sobre 2e5 puntos — np.vectorize                          13.46 ms
erf sobre 2e5 puntos — scipy.special.erf (ufunc real)         1.37 ms


## 7. Para llevar

* La ganancia de vectorizar no es una constante universal — depende de cuánto
  trabajo por elemento hace el bucle interno, y de cuánto overhead de Python se
  elimina al sacarlo. En los experimentos de este cuaderno el rango medido va
  de ~9x (la función de partición, donde ambas versiones ya pasan casi todo su
  tiempo dentro de `exp`) a ~86x (elevar al cuadrado un arreglo, la operación
  con menos trabajo por elemento y por lo tanto la más dominada por el
  intérprete de Python en su versión con bucle).
* `numpy.vectorize` **no es vectorización real** — es una conveniencia de
  broadcasting sobre una función escalar, con el mismo bucle de Python por
  debajo más overhead adicional. En el experimento de la Sección 5 eso no fue
  una sutileza teórica: `np.vectorize` fue medido más lento que el bucle
  explícito que decía reemplazar. Cuando el desempeño importa, la pregunta
  correcta es "¿existe una ufunc real para esto?", no "¿la envuelvo con
  `vectorize`?".
* La vectorización con NumPy cambia tiempo por memoria: el ejemplo de $N$
  cuerpos construye explícitamente un arreglo $O(N^2)$. Ese es exactamente el
  tipo de decisión que la Sesión 5 del curso enseñó a diagnosticar con un
  perfilador antes de tomarla a ciegas.
* Este cuaderno es el punto de partida natural para la Sesión 7: todo lo que
  aquí sigue siendo lento después de vectorizar — como el `np.vectorize` sobre
  `erf`, que no tiene una ufunc de NumPy/SciPy disponible en algún caso más
  exótico — es candidato directo a compilarse con Numba.